In [ ]:
import pandas as pd
from trustifai import Trustifai, MetricContext
from datasets import load_dataset
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
from dotenv import load_dotenv
load_dotenv("creds.env")
import time

import asyncio
from langchain_core.documents import Document

from trustifai.async_pipeline import AsyncTrustifai, evaluate_dataset

In [8]:
#suppress pydantic warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [9]:
dataset = load_dataset("vibrantlabsai/amnesty_qa", "english_v3", split="eval")

In [ ]:
LABEL_MAP_ORDINAL = {
"UNRELIABLE": 0,
"ACCEPTABLE (WITH CAUTION)": 1,
"RELIABLE": 2,
}

LABEL_MAP_BINARY = {
"UNRELIABLE": 0,
"ACCEPTABLE (WITH CAUTION)": 1,
"RELIABLE": 1,
}

def prepare_labels(df: pd.DataFrame):
    df = df.copy()
    df["response_label_ordinal"] = df["response_label"].map(LABEL_MAP_ORDINAL)
    df["response_label_binary"] = df["response_label"].map(LABEL_MAP_BINARY)
    df['ground_truth_label_ordinal'] = df["ground_truth_label"].map(LABEL_MAP_ORDINAL)
    df['ground_truth_label_binary'] = df["ground_truth_label"].map(LABEL_MAP_BINARY)
    return df

In [53]:
df = dataset.to_pandas()

In [ ]:
engine = AsyncTrustifai("config_file.yaml")

In [57]:
def build_contexts(dataset: list[dict], query_col: str, answer_col: str, contexts_col: str) -> list[MetricContext]:
    return [
        MetricContext(
            query=row[query_col],
            answer=row[answer_col],
            documents=[
                Document(page_content=text, metadata={"source": f"doc_{i}"})
                for i, text in enumerate(row[contexts_col])
            ],
        )
        for row in dataset
    ]

In [58]:
async def run_batch():
    RAW_DATASET = df.to_dict(orient="records")
    contexts = build_contexts(RAW_DATASET, query_col="user_input", answer_col="response", contexts_col="retrieved_contexts")

    batch = await evaluate_dataset(
        engine,
        contexts,
        concurrency=3,  # max simultaneous LLM calls
        show_progress=True,   # tqdm bar in terminal or Jupyter
        requests_per_minute=10
    )

    # print("\n" + batch.summary())

    # Access individual results in original order
    for i, result in enumerate(batch.results):
        q = RAW_DATASET[i]["user_input"]
        print(f"  [{i}] Q: {q!r}  →  score={result['score']}  label={result['label']}")

    # Handle any failures gracefully
    if batch.failed:
        print(f"\n  ⚠ {len(batch.failed)} evaluation(s) failed:")
        for f in batch.failed:
            print(f"    index={f['index']}  error={f['error']}")

    return batch

In [59]:
responses = asyncio.run(run_batch())

Evaluating: 100%|██████████| 20/20 [02:08<00:00,  6.41s/sample]

  [0] Q: 'What are the global implications of the USA Supreme Court ruling on abortion?'  →  score=0.79  label=ACCEPTABLE (WITH CAUTION)
  [1] Q: 'Which companies are the main contributors to GHG emissions and their role in global warming according to the Carbon Majors database?'  →  score=0.76  label=ACCEPTABLE (WITH CAUTION)
  [2] Q: 'Which private companies in the Americas are the largest GHG emitters according to the Carbon Majors database?'  →  score=0.59  label=UNRELIABLE
  [3] Q: 'What action did Amnesty International urge its supporters to take in response to the killing of the Ogoni 9?'  →  score=0.74  label=ACCEPTABLE (WITH CAUTION)
  [4] Q: 'What are the recommendations made by Amnesty International to the Special Rapporteur on Human Rights Defenders?'  →  score=0.57  label=UNRELIABLE
  [5] Q: 'Who are the target audience of the two books created by Amnesty International on child rights?'  →  score=0.87  label=RELIABLE
  [6] Q: 'Which right guarantees access to comprehensive

In [61]:
async def run_batch():
    RAW_DATASET = df.to_dict(orient="records")
    contexts = build_contexts(RAW_DATASET, query_col="user_input", answer_col="reference", contexts_col="retrieved_contexts")

    batch = await evaluate_dataset(
        engine,
        contexts,
        concurrency=3,  # max simultaneous LLM calls
        show_progress=True,   # tqdm bar in terminal or Jupyter
        requests_per_minute=10
    )

    # print("\n" + batch.summary())

    # Access individual results in original order
    for i, result in enumerate(batch.results):
        q = RAW_DATASET[i]["user_input"]
        print(f"  [{i}] Q: {q!r}  →  score={result['score']}  label={result['label']}")

    # Handle any failures gracefully
    if batch.failed:
        print(f"\n  ⚠ {len(batch.failed)} evaluation(s) failed:")
        for f in batch.failed:
            print(f"    index={f['index']}  error={f['error']}")

    return batch

In [62]:
ground_truth = asyncio.run(run_batch())

Evaluating: 100%|██████████| 20/20 [02:04<00:00,  6.24s/sample]

  [0] Q: 'What are the global implications of the USA Supreme Court ruling on abortion?'  →  score=0.88  label=RELIABLE
  [1] Q: 'Which companies are the main contributors to GHG emissions and their role in global warming according to the Carbon Majors database?'  →  score=0.88  label=RELIABLE
  [2] Q: 'Which private companies in the Americas are the largest GHG emitters according to the Carbon Majors database?'  →  score=0.9  label=RELIABLE
  [3] Q: 'What action did Amnesty International urge its supporters to take in response to the killing of the Ogoni 9?'  →  score=0.91  label=RELIABLE
  [4] Q: 'What are the recommendations made by Amnesty International to the Special Rapporteur on Human Rights Defenders?'  →  score=0.91  label=RELIABLE
  [5] Q: 'Who are the target audience of the two books created by Amnesty International on child rights?'  →  score=0.87  label=RELIABLE
  [6] Q: 'Which right guarantees access to comprehensive information about past human rights violations, includi

In [73]:
#concat response and ground truth for comparison
final_df = df.copy()
final_df["response_score"] = [responses.results[i]["score"] for i in range(len(responses.results))]
final_df["ground_truth_score"] = [ground_truth.results[i]["score"] for i in range(len(ground_truth.results))]
final_df["response_label"] = [responses.results[i]["label"] for i in range(len(responses.results))]
final_df["ground_truth_label"] = [ground_truth.results[i]["label"] for i in range(len(ground_truth.results))]

In [74]:
final_df = prepare_labels(final_df)

In [75]:
final_df.head()

,user_input,reference,response,retrieved_contexts,response_score,ground_truth_score,response_label,ground_truth_label,response_label_ordinal,response_label_binary,ground_truth_label_ordinal,ground_truth_label_binary
0,What are the global implications of the USA Su...,The global implications of the USA Supreme Cou...,The global implications of the USA Supreme Cou...,"[- In 2022, the USA Supreme Court handed down ...",0.79,0.88,ACCEPTABLE (WITH CAUTION),RELIABLE,1,1,2,1
1,Which companies are the main contributors to G...,"According to the Carbon Majors database, the m...","According to the Carbon Majors database, the m...","[In recent years, there has been increasing pr...",0.76,0.88,ACCEPTABLE (WITH CAUTION),RELIABLE,1,1,2,1
2,Which private companies in the Americas are th...,The largest private companies in the Americas ...,"According to the Carbon Majors database, the l...",[The issue of greenhouse gas emissions has bec...,0.59,0.90,UNRELIABLE,RELIABLE,0,0,2,1
3,What action did Amnesty International urge its...,Amnesty International urged its supporters to ...,Amnesty International urged its supporters to ...,"[In the case of the Ogoni 9, Amnesty Internati...",0.74,0.91,ACCEPTABLE (WITH CAUTION),RELIABLE,1,1,2,1
4,What are the recommendations made by Amnesty I...,The recommendations made by Amnesty Internatio...,Amnesty International made several recommendat...,"[In recent years, Amnesty International has fo...",0.57,0.91,UNRELIABLE,RELIABLE,0,0,2,1


In [ ]:
final_df.to_csv("benchmark_results.csv", index=False)

In [76]:
def compute_metrics(df: pd.DataFrame):
    if df is None:
        raise RuntimeError("Run evaluation first")

    metrics = {}

    def safe_auc(y, s):
        return roc_auc_score(y, s) if len(set(y)) > 1 else None

    def safe_pr(y, s):
        return average_precision_score(y, s) if len(set(y)) > 1 else None

    # Binary detection
    metrics["response_roc_auc"] = safe_auc(
        df["response_label_binary"], df["response_score"]
    )
    metrics["response_pr_auc"] = safe_pr(
        df["response_label_binary"], df["response_score"]
    )

    # Distribution comparison
    distribution_df = pd.DataFrame({
        'Label': list(df["response_label"]) + list(df["ground_truth_label"]),
        'Type': ["LLM"] * len(df) + ["Ground Truth"] * len(df)
    })

    metrics["label_distribution"] = (
        distribution_df.groupby("Label")
            .value_counts()
            .unstack(fill_value=0).T
    )

    return metrics


def generate_report(
    metrics: dict,
    export_path: str = "benchmark_report.md",
):
    """
    Generate a TrustifAI benchmark report (Markdown),
    strictly aligned with the expected reference format.
    """

    def fmt(x):
        return "N/A" if x is None else f"{x:.3f}"

    lines = []

    # --------------------------------------------------
    # Header
    # --------------------------------------------------
    lines.append("# TrustifAI Benchmark Report\n")
    lines.append(f"**Generated on:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

    # --------------------------------------------------
    # Dataset Details
    # --------------------------------------------------
    lines.append("## Dataset Details\n")
    lines.append(
        "This benchmark is conducted using the "
        "[vibrantlabsai/amnesty_qa dataset](https://huggingface.co/datasets/vibrantlabsai/amnesty_qa) from huggingface "
        "which contains question-answer pairs related to human rights "
        "and Amnesty International reports. The dataset includes:\n\n"
        "- 20 ground-truth answers sourced directly from verified Amnesty International documents\n"
        "- 20 LLM-generated answers produced by querying language models\n"
        "- Total of 40 QA pairs evaluated\n\n"
        "The ground truth answers serve as a reliable baseline, while the LLM "
        "answers help assess TrustifAI's ability to detect potential "
        "hallucinations and inaccuracies in model-generated content.\n"
    )

    # --------------------------------------------------
    # What Is Being Evaluated
    # --------------------------------------------------
    lines.append("\n## What Is Being Evaluated?\n")
    lines.append(
        "TrustifAI assigns a **trust score between 0 and 1** to each answer.\n\n"
        "- **High score** → Reliable Answer\n"
        "- **Moderate Score** → Acceptable answer (with caution)\n"
        "- **Low score** → Unreliable (Likely Hallucinated) Answer\n\n"
        "We evaluate TrustifAI on:\n"
        "1. **LLM-generated answers**\n"
        "2. **Ground-truth answers** (known to be correct)\n\n"
        "**Expected behavior:** Ground-truth answers should consistently receive "
        "higher trust scores than LLM answers.\n"
    )

    # --------------------------------------------------
    # Hallucination Detection
    # --------------------------------------------------
    lines.append("\n## Hallucination Detection (Binary Classification)\n")
    lines.append(
        "Labels are mapped as:\n"
        "- **Trustworthy (1)** → RELIABLE, ACCEPTABLE (WITH CAUTION)\n"
        "- **Untrustworthy (0)** → UNRELIABLE\n\n"
        "**Interpretation:**\n"
        "- ROC-AUC → separability between trustworthy vs untrustworthy answers\n"
        "- PR-AUC → robustness under class imbalance\n\n"
        "**Results:**\n"
        "```text\n"
        f"ROC-AUC  : {fmt(metrics.get('response_roc_auc'))}\n"
        f"PR-AUC   : {fmt(metrics.get('response_pr_auc'))}\n"
        "```"
    )

    # --------------------------------------------------
    # Reliability Distribution
    # --------------------------------------------------
    lines.append("\n## Reliability Distribution Comparison\n")
    lines.append(
        "A healthy system should assign:\n"
        "- More **RELIABLE** labels to **Ground Truth**\n"
        "- More **UNRELIABLE** labels to **LLM answers**\n\n"
    )
    lines.append("**Results:**\n")

    dist = metrics.get("label_distribution")
    if dist is not None:
        lines.append("```text")
        lines.append(dist.to_string())
        lines.append("```")
    else:
        lines.append("```text\nDistribution table not available.\n```")

    # --------------------------------------------------
    # Verdict
    # --------------------------------------------------
    lines.append(
        "\n## Verdict\n\n"
        "TrustifAI demonstrates **meaningful separation** between grounded and "
        "hallucinated answers. Ground-truth responses consistently receive "
        "higher trust scores, indicating:\n\n"
        "- Effective hallucination detection\n"
        "- Reasonable score calibration\n"
        "- Practical usefulness in RAG evaluation pipelines\n"
    )

    report = "\n".join(lines)

    with open(export_path, "w") as f:
        f.write(report)

    print("Benchmark report exported.")

In [77]:
metric_df = compute_metrics(final_df)

In [ ]:
generate_report(metric_df)

Benchmark report exported.
